# Seam Diagnostics — Mesh Crack Investigation & Fix

All anonymization methods produce identical dark "crack" lines on the face.
The cracks appear at the **same positions** regardless of method, proving the
cause is the **mesh structure** — duplicate vertices at patch boundaries from
Einstar's multi-patch stitching.

This notebook:
1. Loads the scan, clicks Nz, auto-detects landmarks
2. Visualizes seam lines on the original mesh
3. Confirms seam lines match crack locations
4. Tests a vertex merge fix
5. Quantifies the merge
6. Validates that colors survive the merge

In [2]:
import numpy as np
import pyvista as pv
import trimesh
from scipy.spatial import KDTree

import cedalion
import cedalion.io
import cedalion.plots
from cedalion import units
from cedalion.dataclasses import VTKSurface
from cedalion.geometry.photogrammetry.anonymization import (
    detect_landmarks_from_nasion,
    anonymize_scan,
    AnonymizationConfig,
    AnonymizationMethod,
)

pv.set_jupyter_backend("server")

# Load scan
SUBJECT_NUMBER = 12
SCANS_FOLDER = "/home/ma7/BA/PG_Subjects11-15"
scan_path = f"{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj"
surface = cedalion.io.read_einstar_obj(scan_path)

mesh = surface.mesh
print(f"Loaded: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces")

Loaded: 624,121 vertices, 1,169,802 faces


## 1b. Click Nz on the Mesh

Click the nasion (bridge of the nose, between the eyebrows) on the mesh.
The clicked point is stored in `nz_store`.

In [3]:
nz_store = {}

def on_pick(point):
    """Callback for surface point picking."""
    nz_store["point"] = np.array(point)
    print(f"Nz selected at: X={point[0]:.1f}, Y={point[1]:.1f}, Z={point[2]:.1f}")

# Use a pop-up window — callbacks don't work with the jupyter server backend
pvplt = pv.Plotter(notebook=False)
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.enable_surface_point_picking(
    callback=on_pick,
    left_clicking=True,
    show_point=True,
    point_size=20,
    color="red",
    show_message="Left-click Nz (nasion), then close the window",
)
pvplt.add_text("Left-click Nz (nasion), then close window", position="upper_left", font_size=14)
pvplt.show()

Nz selected at: X=113.7, Y=60.6, Z=523.9
Nz selected at: X=111.1, Y=150.8, Z=427.8


In [4]:
# If clicking didn't work, manually set Nz coordinates here.
# Hover over the nasion in the plot above to read coordinates, then uncomment:
# nz_store["point"] = np.array([X, Y, Z])

if "point" in nz_store:
    print(f"Nz stored: {nz_store['point']}")
else:
    print("Nz NOT stored yet. Either click above or set manually in this cell.")

Nz stored: [111.11239585 150.77507346 427.75947951]


## 1c. Auto-Detect Remaining Landmarks

From the clicked Nz, the algorithm detects Cz (vertex), Iz (inion),
LPA (left preauricular), and RPA (right preauricular) using mesh geometry.

In [5]:
nz_point = nz_store["point"]
print(f"Nz position: {nz_point}")

landmarks = detect_landmarks_from_nasion(surface, nz_point)

print(f"\nDetected landmarks:")
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    print(f"  {label}: X={pos[0]:7.1f}, Y={pos[1]:7.1f}, Z={pos[2]:7.1f}")

Nz position: [111.11239585 150.77507346 427.75947951]

Detected landmarks:
  Nz: X=  111.1, Y=  150.8, Z=  427.8
  Iz: X=  118.4, Y=  -46.2, Z=  439.6
  Cz: X=  204.9, Y=   42.6, Z=  437.6
  LPA: X=  102.4, Y=   47.4, Z=  532.4
  RPA: X=   80.0, Y=   32.6, Z=  350.6


## 2. Visualize Seam Lines on Original Mesh

Boundary edges belong to only 1 face (instead of the normal 2). These mark
patch boundaries where the Einstar stitcher left topologically disconnected seams.

In [6]:
# Find boundary edges: edges that belong to only 1 face
# trimesh tracks this via edges_face — boundary edges have -1 for missing face
from collections import Counter

# Count how many faces each edge belongs to
edge_to_face_count = Counter()
for face in mesh.faces:
    edges_in_face = [
        tuple(sorted((face[0], face[1]))),
        tuple(sorted((face[1], face[2]))),
        tuple(sorted((face[0], face[2]))),
    ]
    for e in edges_in_face:
        edge_to_face_count[e] += 1

boundary_edges = [e for e, count in edge_to_face_count.items() if count == 1]
print(f"Boundary edges: {len(boundary_edges):,}")
print(f"Total edges: {len(edge_to_face_count):,}")
print(f"Boundary fraction: {100 * len(boundary_edges) / len(edge_to_face_count):.1f}%")

# Extract boundary edge midpoints for visualization
boundary_edge_arr = np.array(boundary_edges)
midpoints = 0.5 * (mesh.vertices[boundary_edge_arr[:, 0]] + mesh.vertices[boundary_edge_arr[:, 1]])

# Build line segments for pyvista
boundary_verts = mesh.vertices[boundary_edge_arr.ravel()]
n_edges = len(boundary_edges)
lines = np.column_stack([
    np.full(n_edges, 2),          # each line has 2 points
    np.arange(0, 2 * n_edges, 2), # start indices
    np.arange(1, 2 * n_edges, 2), # end indices
]).ravel()
boundary_polydata = pv.PolyData(boundary_verts, lines=lines)

# Render mesh with boundary edges in red
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface, opacity=0.6)
pvplt.add_mesh(boundary_polydata, color="red", line_width=2, opacity=0.8)
pvplt.add_text(
    f"Seam lines (boundary edges): {len(boundary_edges):,}",
    position="upper_left", font_size=14,
)
pvplt.show()

Boundary edges: 72,350
Total edges: 1,790,870
Boundary fraction: 4.0%


Widget(value='<iframe src="http://localhost:34739/index.html?ui=P_0x7f7f296f65d0_0&reconnect=auto" class="pyvi…

## 3. Confirm: Seams Match Crack Locations

Run smoothing **without** vertex merging (the old code path) to reproduce the
cracks, then compare side-by-side with the seam overlay. The cracks should
line up with the red seam lines.

In [7]:
# Run heat diffusion smoothing WITHOUT merging to reproduce cracks.
# This manually calls the internal functions, bypassing the merge step
# that anonymize_scan() now applies automatically.
from cedalion.geometry.photogrammetry.anonymization.anonymizer import (
    _heat_diffusion_smooth,
    _get_protected_vertex_indices,
    apply_boundary_transition,
    smooth_region_selective,
)
from cedalion.geometry.photogrammetry.anonymization import (
    detect_facial_landmarks,
    get_facial_region_mask,
)
from copy import deepcopy
import cedalion.dataclasses as cdc

# Compute facial mask on original (unmerged) mesh
facial_lm = detect_facial_landmarks(surface, landmarks)
facial_mask = get_facial_region_mask(
    surface=surface,
    facial_landmarks=facial_lm,
    protected_points=landmarks,
    protection_radius=15.0 * units.mm,
)

# Get protected indices on original mesh
protected_indices = _get_protected_vertex_indices(
    mesh, landmarks, protection_radius=15.0,
)

# Run Taubin smoothing directly on the UNMERGED mesh (old code path → cracks)
# Use 300 iterations with uniform Laplacian for strong, visible smoothing
cracked_mesh = smooth_region_selective(
    mesh=mesh,
    mask=facial_mask,
    protected_indices=protected_indices,
    iterations=300,
    lamb=0.5,
    mu=-0.53,
    use_cotangent=False,  # uniform Laplacian — no auto-scaling dampening
)
cracked_mesh = apply_boundary_transition(
    mesh=cracked_mesh,
    mask=facial_mask,
    transition_width=10.0,
    protected_indices=protected_indices,
    iterations=10,
    lamb=0.3,
)

cracked_surface = cdc.TrimeshSurface(
    mesh=cracked_mesh,
    crs=surface.crs,
    units=surface.units,
)

# Check displacement
disp_cracked = np.linalg.norm(cracked_mesh.vertices - mesh.vertices, axis=1)
face_disp = disp_cracked[facial_mask]
print(f"Cracked result (no merge, 300 iter uniform Taubin):")
print(f"  Max displacement:  {disp_cracked.max():.2f} mm")
print(f"  Mean face disp:    {face_disp.mean():.2f} mm")
print(f"  Median face disp:  {np.median(face_disp):.2f} mm")

# Side-by-side: original + seam overlay vs cracked result
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface, opacity=0.6)
pvplt.add_mesh(boundary_polydata, color="red", line_width=2, opacity=0.8)
pvplt.add_text("Original + Seam Lines", position="upper_left", font_size=12)

pvplt.subplot(0, 1)
cedalion.plots.plot_surface(pvplt, cracked_surface, opacity=1.0)
pvplt.add_text("300 iter Taubin, no merge (cracks)", position="upper_left", font_size=12)

pvplt.link_views()
pvplt.show()

Cracked result (no merge, 300 iter uniform Taubin):
  Max displacement:  3.42 mm
  Mean face disp:    0.22 mm
  Median face disp:  0.16 mm


Widget(value='<iframe src="http://localhost:34739/index.html?ui=P_0x7f7ef9555750_1&reconnect=auto" class="pyvi…

## 4. Test Fix: Merge Vertices (via anonymize_scan)

`anonymize_scan()` now automatically merges duplicate vertices before smoothing.
Run it normally and compare against the cracked result from cell 3.

In [35]:
# Also do the manual merge here so cells 5 & 6 can use the stats
def merge_seam_vertices(mesh, tol=0.01):
    """Merge spatially coincident vertices into single vertices."""
    verts = mesh.vertices
    n = len(verts)

    tree = KDTree(verts)
    pairs = tree.query_pairs(r=tol)

    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for a, b in pairs:
        union(a, b)

    for i in range(n):
        parent[i] = find(i)

    unique_roots, inverse = np.unique(parent, return_inverse=True)
    old_to_new = inverse
    n_new = len(unique_roots)

    new_to_old = [[] for _ in range(n_new)]
    for old_idx in range(n):
        new_to_old[old_to_new[old_idx]].append(old_idx)

    new_verts = np.zeros((n_new, 3))
    for new_idx in range(n_new):
        old_indices = new_to_old[new_idx]
        new_verts[new_idx] = verts[old_indices].mean(axis=0)

    new_faces = old_to_new[mesh.faces]
    valid = (
        (new_faces[:, 0] != new_faces[:, 1])
        & (new_faces[:, 1] != new_faces[:, 2])
        & (new_faces[:, 0] != new_faces[:, 2])
    )
    new_faces = new_faces[valid]

    new_colors = None
    if hasattr(mesh.visual, 'vertex_colors'):
        try:
            old_colors = mesh.visual.to_color().vertex_colors
            new_colors = np.zeros((n_new, 4), dtype=np.float64)
            for new_idx in range(n_new):
                old_indices = new_to_old[new_idx]
                new_colors[new_idx] = old_colors[old_indices].mean(axis=0)
            new_colors = new_colors.astype(np.uint8)
        except Exception:
            pass

    merged_mesh = trimesh.Trimesh(vertices=new_verts, faces=new_faces, process=False)
    if new_colors is not None:
        merged_mesh.visual.vertex_colors = new_colors

    return merged_mesh, old_to_new, new_to_old


merged_mesh, old_to_new, new_to_old = merge_seam_vertices(mesh, tol=0.01)
merged_surface = cdc.TrimeshSurface(mesh=merged_mesh, crs=surface.crs, units=surface.units)

print(f"Before merge: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces")
print(f"After merge:  {len(merged_mesh.vertices):,} vertices, {len(merged_mesh.faces):,} faces")
print(f"Vertices removed: {len(mesh.vertices) - len(merged_mesh.vertices):,}")
print(f"Faces removed (degenerate): {len(mesh.faces) - len(merged_mesh.faces):,}")

Before merge: 624,121 vertices, 1,169,802 faces
After merge:  584,684 vertices, 1,169,260 faces
Vertices removed: 39,437
Faces removed (degenerate): 542


In [36]:
# Run anonymize_scan() which now includes automatic vertex merging
# Use SMOOTH with uniform Laplacian and 300 iterations for strong visible effect
config = AnonymizationConfig(
    method=AnonymizationMethod.SMOOTH,
    smoothing_iterations=300,
    smoothing_lambda=0.5,
    smoothing_mu=-0.53,
    use_cotangent=False,  # uniform Laplacian — avoids cotangent auto-scaling
)
result_fixed = anonymize_scan(
    surface=surface,
    anatomical_landmarks=landmarks,
    config=config,
    interactive=False,
    validate=True,
)

disp = result_fixed.vertex_displacements
face_disp = disp[result_fixed.facial_mask]
print(f"Fixed result (with merge, 300 iter uniform Taubin):")
print(f"  Max displacement:  {disp.max():.2f} mm")
print(f"  Mean face disp:    {face_disp.mean():.2f} mm")
print(f"  Median face disp:  {np.median(face_disp):.2f} mm")

# Side-by-side: cracked (no merge) vs fixed (with merge)
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, cracked_surface, opacity=1.0)
pvplt.add_text("Without merge (cracks)", position="upper_left", font_size=12)

pvplt.subplot(0, 1)
cedalion.plots.plot_surface(pvplt, result_fixed.anonymized_surface, opacity=1.0)
pvplt.add_text("With merge (fixed)", position="upper_left", font_size=12)

pvplt.link_views()
pvplt.show()

Fixed result (with merge, 300 iter uniform Taubin):
  Max displacement:  1.34 mm
  Mean face disp:    0.17 mm
  Median face disp:  0.15 mm


Widget(value='<iframe src="http://localhost:45781/index.html?ui=P_0x7f13f5d2c990_13&reconnect=auto" class="pyv…

## 5. Quantify the Merge

Print detailed statistics on the merge operation and verify mesh integrity.

In [28]:
# Count connected components before and after
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

def count_components(m):
    """Count connected components in a trimesh."""
    n = len(m.vertices)
    edges = m.edges_unique
    rows = np.concatenate([edges[:, 0], edges[:, 1]])
    cols = np.concatenate([edges[:, 1], edges[:, 0]])
    data = np.ones(len(rows))
    A = csr_matrix((data, (rows, cols)), shape=(n, n))
    n_comp, _ = connected_components(A, directed=False)
    return n_comp

n_comp_before = count_components(mesh)
n_comp_after = count_components(merged_mesh)

# Count boundary edges after merge
edge_count_after = Counter()
for face in merged_mesh.faces:
    for i in range(3):
        e = tuple(sorted((face[i], face[(i + 1) % 3])))
        edge_count_after[e] += 1
boundary_after = sum(1 for c in edge_count_after.values() if c == 1)

print("=== Merge Statistics ===")
print(f"Vertices:   {len(mesh.vertices):>10,} -> {len(merged_mesh.vertices):>10,}  ({len(mesh.vertices) - len(merged_mesh.vertices):,} removed)")
print(f"Faces:      {len(mesh.faces):>10,} -> {len(merged_mesh.faces):>10,}  ({len(mesh.faces) - len(merged_mesh.faces):,} degenerate removed)")
print(f"Components: {n_comp_before:>10,} -> {n_comp_after:>10,}")
print(f"Boundary edges: {len(boundary_edges):>7,} -> {boundary_after:>7,}")
print()

# Check for degenerate faces in merged mesh
areas = merged_mesh.area_faces
degenerate = (areas < 1e-10).sum()
print(f"Degenerate faces (area < 1e-10): {degenerate}")
print(f"Mesh is watertight: {merged_mesh.is_watertight}")
print(f"Total surface area: {mesh.area:.1f} mm^2 (before) vs {merged_mesh.area:.1f} mm^2 (after)")

=== Merge Statistics ===
Vertices:      624,121 ->    584,684  (39,437 removed)
Faces:       1,169,802 ->  1,169,260  (542 degenerate removed)
Components:      3,715 ->         28
Boundary edges:  72,350 ->       0

Degenerate faces (area < 1e-10): 0
Mesh is watertight: True
Total surface area: 396396.6 mm^2 (before) vs 396396.6 mm^2 (after)


## 6. Test: Does Merging Break Vertex Colors?

Compare original vs merged mesh appearance (without any smoothing) to verify
that color averaging at seam positions preserves visual appearance.

In [19]:
# Render original vs merged mesh WITHOUT anonymization
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.add_text("Original", position="upper_left", font_size=12)

pvplt.subplot(0, 1)
cedalion.plots.plot_surface(pvplt, merged_surface, opacity=1.0)
pvplt.add_text("Merged (no smoothing)", position="upper_left", font_size=12)

pvplt.link_views()
pvplt.show()

# Quantify color difference at seam vertices
try:
    orig_colors = mesh.visual.to_color().vertex_colors[:, :3].astype(float)
    merged_colors = merged_mesh.visual.to_color().vertex_colors[:, :3].astype(float)

    # For each merged vertex, compare its color to the average of original colors
    max_diffs = []
    for new_idx in range(len(merged_mesh.vertices)):
        old_indices = new_to_old[new_idx]
        if len(old_indices) > 1:  # only check actually merged vertices
            old_cols = orig_colors[old_indices]
            spread = old_cols.max(axis=0) - old_cols.min(axis=0)
            max_diffs.append(spread.max())

    if max_diffs:
        max_diffs = np.array(max_diffs)
        print(f"\nColor spread at merged vertices (0-255 scale):")
        print(f"  Max spread:    {max_diffs.max():.1f}")
        print(f"  Mean spread:   {max_diffs.mean():.1f}")
        print(f"  Median spread: {np.median(max_diffs):.1f}")
        print(f"  Vertices with spread > 10: {(max_diffs > 10).sum()} / {len(max_diffs)}")
    else:
        print("No merged vertices found.")
except Exception as e:
    print(f"Could not compare colors: {e}")

Widget(value='<iframe src="http://localhost:45781/index.html?ui=P_0x7f13f5d2fb50_7&reconnect=auto" class="pyvi…

Could not compare colors: 'ColorVisuals' object has no attribute 'to_color'
